In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/__results__.html
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/__huggingface_repos__.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/__notebook__.ipynb
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/__output__.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/custom.css
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/config.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/training_args.bin
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/tokenizer.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/tokenizer_config.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/model.safetensors
/kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm/checkpoint-12749/config.json
/kaggle/input/notebooks/aabdollahii/finetuning-on-wi

In [3]:
from pathlib import Path
import pandas as pd

# Paths
PROBE_DIR = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "test-on-fabertwikifarsi-making-testset-sec6/factual_probe"
)

PROBE_FILE = PROBE_DIR / "factual_probe_dataset.csv"
REJECTED_FILE = PROBE_DIR / "factual_probe_rejected.csv"
STATS_FILE = PROBE_DIR / "factual_probe_stats.csv"

TRAINED_MODEL_PATH = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "finetuning-on-wikifarsi/fabert_kg_mlm"
)

# Load CSV files
probe_df = pd.read_csv(PROBE_FILE, encoding="utf-8-sig")
rejected_df = pd.read_csv(REJECTED_FILE, encoding="utf-8-sig")
stats_df = pd.read_csv(STATS_FILE, encoding="utf-8-sig")

print("=" * 80)
print("FILE PATHS")
print("=" * 80)
print("Probe dataset:", PROBE_FILE)
print("Rejected dataset:", REJECTED_FILE)
print("Statistics:", STATS_FILE)
print("Trained model:", TRAINED_MODEL_PATH)

print("\n" + "=" * 80)
print("PROBE DATASET: BASIC INFORMATION")
print("=" * 80)
print("Shape:", probe_df.shape)
print("Rows:", len(probe_df))
print("Columns:", probe_df.columns.tolist())

print("\nData types:")
print(probe_df.dtypes)

print("\nFirst 10 rows:")
display(probe_df.head(10))

print("\nRandom 10 rows:")
display(
    probe_df.sample(
        n=min(10, len(probe_df)),
        random_state=42,
    )
)

print("\n" + "=" * 80)
print("MISSING VALUES")
print("=" * 80)

missing_summary = pd.DataFrame({
    "missing_count": probe_df.isna().sum(),
    "missing_percent": (
        probe_df.isna().mean() * 100
    ).round(2),
})

display(
    missing_summary.sort_values(
        "missing_count",
        ascending=False,
    )
)

print("\n" + "=" * 80)
print("DUPLICATES")
print("=" * 80)
print(
    "Fully duplicated rows:",
    probe_df.duplicated().sum(),
)

duplicate_key_columns = [
    column
    for column in ["subject", "predicate", "object", "prompt"]
    if column in probe_df.columns
]

if duplicate_key_columns:
    print(
        f"Duplicates based on {duplicate_key_columns}:",
        probe_df.duplicated(
            subset=duplicate_key_columns
        ).sum(),
    )

print("\n" + "=" * 80)
print("CATEGORICAL DISTRIBUTIONS")
print("=" * 80)

categorical_columns = [
    "predicate",
    "template_id",
    "template_type",
    "split_type",
    "valid",
    "object_token_count",
]

for column in categorical_columns:
    if column in probe_df.columns:
        print(f"\n--- {column} ---")

        distribution = (
            probe_df[column]
            .value_counts(dropna=False)
            .rename_axis(column)
            .reset_index(name="count")
        )

        distribution["percent"] = (
            distribution["count"] /
            len(probe_df) * 100
        ).round(2)

        display(distribution)

print("\n" + "=" * 80)
print("MOST FREQUENT ANSWERS")
print("=" * 80)

if "object" in probe_df.columns:
    object_distribution = (
        probe_df["object"]
        .value_counts(dropna=False)
        .head(30)
        .rename_axis("object")
        .reset_index(name="count")
    )

    object_distribution["percent"] = (
        object_distribution["count"] /
        len(probe_df) * 100
    ).round(2)

    display(object_distribution)

print("\n" + "=" * 80)
print("OBJECT DISTRIBUTION BY PREDICATE")
print("=" * 80)

if {"predicate", "object"}.issubset(probe_df.columns):
    predicate_summary = (
        probe_df
        .groupby("predicate")
        .agg(
            rows=("object", "size"),
            unique_subjects=("subject", "nunique"),
            unique_objects=("object", "nunique"),
        )
        .reset_index()
        .sort_values("rows", ascending=False)
    )

    predicate_summary["most_common_object"] = (
        probe_df
        .groupby("predicate")["object"]
        .agg(lambda values: values.value_counts().index[0])
        .reindex(predicate_summary["predicate"])
        .values
    )

    predicate_summary["most_common_object_count"] = (
        probe_df
        .groupby("predicate")["object"]
        .agg(lambda values: values.value_counts().iloc[0])
        .reindex(predicate_summary["predicate"])
        .values
    )

    predicate_summary["top_object_share_percent"] = (
        predicate_summary["most_common_object_count"] /
        predicate_summary["rows"] * 100
    ).round(2)

    display(predicate_summary)

print("\n" + "=" * 80)
print("PROMPT VALIDATION")
print("=" * 80)

if "prompt" in probe_df.columns:
    mask_token = "[MASK]"

    mask_counts = probe_df["prompt"].astype(str).str.count(
        r"\[MASK\]"
    )

    print("Rows with exactly one [MASK]:", (mask_counts == 1).sum())
    print("Rows with no [MASK]:", (mask_counts == 0).sum())
    print("Rows with multiple [MASK]:", (mask_counts > 1).sum())

if {"prompt", "object"}.issubset(probe_df.columns):
    answer_leak = probe_df.apply(
        lambda row: str(row["object"]) in str(row["prompt"]),
        axis=1,
    )

    print("Rows with gold-answer leakage:", answer_leak.sum())

    if answer_leak.any():
        display(
            probe_df.loc[
                answer_leak,
                ["subject", "predicate", "object", "prompt"]
            ].head(20)
        )

print("\n" + "=" * 80)
print("REJECTED DATA")
print("=" * 80)
print("Shape:", rejected_df.shape)
print("Columns:", rejected_df.columns.tolist())

display(rejected_df.head(10))

if "rejection_reason" in rejected_df.columns:
    rejected_summary = (
        rejected_df["rejection_reason"]
        .value_counts(dropna=False)
        .rename_axis("rejection_reason")
        .reset_index(name="count")
    )

    rejected_summary["percent"] = (
        rejected_summary["count"] /
        len(rejected_df) * 100
    ).round(2)

    display(rejected_summary)

print("\n" + "=" * 80)
print("SAVED STATISTICS FILE")
print("=" * 80)
print("Shape:", stats_df.shape)
print("Columns:", stats_df.columns.tolist())
display(stats_df)

print("\n" + "=" * 80)
print("TRAINED MODEL FILES")
print("=" * 80)

if TRAINED_MODEL_PATH.exists():
    for file_path in sorted(TRAINED_MODEL_PATH.iterdir()):
        if file_path.is_file():
            size_mb = file_path.stat().st_size / (1024 ** 2)
            print(f"{file_path.name:<30} {size_mb:>10.2f} MB")
        else:
            print(f"{file_path.name:<30} <directory>")
else:
    print("Model path was not found:", TRAINED_MODEL_PATH)


/tmp/ipykernel_58/1203061752.py:21: DtypeWarning: Columns (6,7,8,9,10,11,12,13,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  rejected_df = pd.read_csv(REJECTED_FILE, encoding="utf-8-sig")


FILE PATHS
Probe dataset: /kaggle/input/notebooks/aabdollahii/test-on-fabertwikifarsi-making-testset-sec6/factual_probe/factual_probe_dataset.csv
Rejected dataset: /kaggle/input/notebooks/aabdollahii/test-on-fabertwikifarsi-making-testset-sec6/factual_probe/factual_probe_rejected.csv
Statistics: /kaggle/input/notebooks/aabdollahii/test-on-fabertwikifarsi-making-testset-sec6/factual_probe/factual_probe_stats.csv
Trained model: /kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm

PROBE DATASET: BASIC INFORMATION
Shape: (1206, 15)
Rows: 1206
Columns: ['fact_id', 'subject', 'predicate', 'object', 'prompt', 'template_id', 'template_type', 'gold_answer', 'gold_token_id', 'object_token_count', 'object_tokens', 'split_type', 'source', 'valid', 'validation_error']

Data types:
fact_id                object
subject                object
predicate              object
object                 object
prompt                 object
template_id            object
template_type      

,fact_id,subject,predicate,object,prompt,template_id,template_type,gold_answer,gold_token_id,object_token_count,object_tokens,split_type,source,valid,validation_error
0,fact_000001,محمد پورستار,محل تولد,اردبیل,زادگاه محمد پورستار [MASK] است.,birthplace_03,paraphrased,اردبیل,10050,1,"[""اردبیل""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
1,fact_000002,بخش ایرندگان,زبان,بلوچی,زبان بخش ایرندگان [MASK] است.,language_01,familiar,بلوچی,31449,1,"[""بلوچی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
2,fact_000003,پرم چوپرا,محل تولد,لاهور,محل تولد پرم چوپرا [MASK] است.,birthplace_01,familiar,لاهور,49461,1,"[""لاهور""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
3,fact_000004,یاروسلاو سایفرت,ملیت,چکی,یاروسلاو سایفرت فردی [MASK] است.,nationality_03,paraphrased,چکی,36644,1,"[""چکی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
4,fact_000005,سه برخوانی,کشور,تهران,کشور محل قرارگیری سه برخوانی، [MASK] است.,country_02,paraphrased,تهران,3148,1,"[""تهران""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
5,fact_000006,پاستورا سولر,ملیت,اسپانیایی,ملیت پاستورا سولر [MASK] است.,nationality_01,familiar,اسپانیایی,17013,1,"[""اسپانیایی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
6,fact_000007,اسکامپولو ۵۳ (فیلم ۱۹۵۳),زبان,ایتالیایی,زبان اسکامپولو ۵۳ (فیلم ۱۹۵۳) [MASK] است.,language_01,familiar,ایتالیایی,13857,1,"[""ایتالیایی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
7,fact_000008,قباد آذرآیین,محل تولد,مسجدسلیمان,محل تولد قباد آذرآیین [MASK] است.,birthplace_01,familiar,مسجدسلیمان,32192,1,"[""مسجدسلیمان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
8,fact_000009,ژرژ لامپن,ملیت,فرانسه,ژرژ لامپن فردی [MASK] است.,nationality_03,paraphrased,فرانسه,5916,1,"[""فرانسه""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
9,fact_000010,کشکش,استان,گیلان,کشکش در استان [MASK] قرار دارد.,province_01,familiar,گیلان,8188,1,"[""گیلان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN



Random 10 rows:


,fact_id,subject,predicate,object,prompt,template_id,template_type,gold_answer,gold_token_id,object_token_count,object_tokens,split_type,source,valid,validation_error
101,fact_000111,بی نهر علیا,استان,کرمانشاه,استان محل قرارگیری بی نهر علیا، [MASK] است.,province_02,paraphrased,کرمانشاه,8746,1,"[""کرمانشاه""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
260,fact_000280,آقای نابغه (فیلم),کشور,هند,آقای نابغه (فیلم) در کشور [MASK] قرار دارد.,country_01,familiar,هند,4837,1,"[""هند""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
777,fact_000836,آکادمی بررا,کشور,ایتالیا,آکادمی بررا متعلق به کشور [MASK] است.,country_03,paraphrased,ایتالیا,8564,1,"[""ایتالیا""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
109,fact_000119,محمود پیراسته,محل تولد,بوشهر,زادگاه محمود پیراسته [MASK] است.,birthplace_03,paraphrased,بوشهر,9191,1,"[""بوشهر""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
649,fact_000699,خرگی,استان,هرمزگان,استان محل قرارگیری خرگی، [MASK] است.,province_02,paraphrased,هرمزگان,14931,1,"[""هرمزگان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
735,fact_000789,فرانسواز راشمول,محل تولد,نانسی,زادگاه فرانسواز راشمول [MASK] است.,birthplace_03,paraphrased,نانسی,39373,1,"[""نانسی""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
332,fact_000355,آوین سفلی,استان,هرمزگان,آوین سفلی از مناطق استان [MASK] است.,province_03,paraphrased,هرمزگان,14931,1,"[""هرمزگان""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
49,fact_000054,اخطار (فیلم ۲۰۱۸),کشور,اسپانیا,اخطار (فیلم ۲۰۱۸) متعلق به کشور [MASK] است.,country_03,paraphrased,اسپانیا,9563,1,"[""اسپانیا""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
461,fact_000494,علی عباس قدیروف,محل تولد,باکو,علی عباس قدیروف در [MASK] متولد شد.,birthplace_02,paraphrased,باکو,19635,1,"[""باکو""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN
980,fact_001051,امامزاده بزم,استان,فارس,امامزاده بزم از مناطق استان [MASK] است.,province_03,paraphrased,فارس,4872,1,"[""فارس""]",seen,/kaggle/input/notebooks/aabdollahii/persian-wi...,True,NaN


...

,subject,predicate,object,rejection_reason,object_token_count,gold_token_id,object_tokens,fact_id,prompt,template_id,template_type,gold_answer,split_type,source,valid,validation_error
0,سعدی,نام,سعدی شیرازی,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,سعدی,زمینه فعالیت,شعر و نثر فارسی,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,سعدی,تاریخ تولد,بین ۵۸۵ تا ۶۱۵ هجری قمری,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,سعدی,تاریخ مرگ,بین ۶۹۰ تا ۶۹۵,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,سعدی,محل مرگ,شیراز، ایران,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,سعدی,محل زندگی,شیراز، بغداد,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,سعدی,مدفن,سعدیه، شیراز,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,سعدی,در زمان حکومت,اتابکان فارس، عباسیان، خوارزمشاهیان، ایلخانان,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,سعدی,اتفاقات مهم,حملهٔ مغول به ایران,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,سعدی,لقب,افصح المتکلمین، استاد سخن، پادشاه سخن، شیخ اجلّ,unsupported_predicate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,rejection_reason,count,percent
0,unsupported_predicate,1386001,80.05
1,numeric_object,262732,15.18
2,multi_token_object,70889,4.09
3,url_object,9717,0.56
4,empty_object,1862,0.11
5,NaN,90,0.01
6,object_encoding_problem,41,0.00



SAVED STATISTICS FILE
Shape: (14800, 11)
Columns: ['predicate', 'raw_count', 'clean_candidate_count', 'single_token_count', 'multi_token_count', 'sampled_fact_count', 'final_probe_count', 'unique_objects', 'most_common_object', 'most_common_object_count', 'most_common_object_share']


,predicate,raw_count,clean_candidate_count,single_token_count,multi_token_count,sampled_fact_count,final_probe_count,unique_objects,most_common_object,most_common_object_count,most_common_object_share
0,(58بخش)num_episodes,1,0,0,0,0,0,0,NaN,0,0.0
1,(IATA,1,0,0,0,0,0,0,NaN,0,0.0
2,(unranked),1,0,0,0,0,0,0,NaN,0,0.0
3,(نام محلی,1,0,0,0,0,0,0,NaN,0,0.0
4,- ! bgcolor,1,0,0,0,0,0,0,NaN,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
14795,۴داده۵,1,0,0,0,0,0,0,NaN,0,0.0
14796,۴داده۶,1,0,0,0,0,0,0,NaN,0,0.0
14797,۴داده۷,1,0,0,0,0,0,0,NaN,0,0.0
14798,۷۰(تاکنون),1,0,0,0,0,0,0,NaN,0,0.0



TRAINED MODEL FILES
checkpoint-12000               <directory>
checkpoint-12749               <directory>
config.json                          0.00 MB
model.safetensors                  474.93 MB
tokenizer.json                       1.30 MB
tokenizer_config.json                0.00 MB
training_args.bin                    0.00 MB


# testing phase

In [4]:
from pathlib import Path
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROBE_FILE = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "test-on-fabertwikifarsi-making-testset-sec6/"
    "factual_probe/factual_probe_dataset.csv"
)

BASELINE_MODEL_PATH = "sbunlp/fabert"

KG_MODEL_PATH = Path(
    "/kaggle/input/notebooks/aabdollahii/"
    "finetuning-on-wikifarsi/fabert_kg_mlm"
)

OUTPUT_DIR = Path("/kaggle/working/factual_probe_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
MAX_LENGTH = 128
TOP_K_TO_SAVE = 10

print("Device:", DEVICE)
print("Probe file exists:", PROBE_FILE.exists())
print("KG model path exists:", KG_MODEL_PATH.exists())
print("KG model path:", KG_MODEL_PATH)


Device: cpu
Probe file exists: True
KG model path exists: True
KG model path: /kaggle/input/notebooks/aabdollahii/finetuning-on-wikifarsi/fabert_kg_mlm
